# Technical Screening and Feasibility Assessment


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\figures
NeqSim Java bridge available: True


## Flow Assurance Gate Matrix


In [2]:
threats = ["Hydrate", "Wax", "Asphaltene", "CO2 corrosion", "Scale"]
concepts = ["Tieback gas", "Oil satellite", "High-CO2 gas"]
risk = np.array([[4, 1, 1, 3, 2], [4, 3, 3, 2, 3], [3, 1, 1, 5, 2]])
fig, ax = plt.subplots(figsize=(9, 4.8))
image = ax.imshow(risk, cmap="YlOrRd", vmin=1, vmax=5)
ax.set_xticks(range(len(threats)), labels=threats, rotation=25, ha="right")
ax.set_yticks(range(len(concepts)), labels=concepts)
for i in range(risk.shape[0]):
    for j in range(risk.shape[1]):
        ax.text(j, i, str(risk[i, j]), ha="center", va="center")
ax.set_title("Screening Risk Matrix (1=low, 5=high)")
fig.colorbar(image, ax=ax, label="Risk score")
fig.savefig(FIGURES_DIR / "ch11_screening_flow_assurance_matrix.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch11_screening_flow_assurance_matrix.png).** *Observation.* Hydrate risk dominates the cold tieback cases, while corrosion dominates the high-CO2 gas case. *Mechanism.* Cold subsea lines favor hydrate stability, whereas high acid-gas partial pressure drives material and inhibitor requirements. *Implication.* The same concept label can fail for different technical reasons. *Recommendation.* Carry the leading threat forward as a named assumption in concept ranking and cost contingency.


## Artificial-Lift Screening


In [3]:
methods = ["Flowing", "Gas lift", "ESP", "Subsea boost"]
scores = np.array([52, 86, 68, 74])
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(methods, scores, color="#3b8ea5")
ax.set_ylim(0, 100)
ax.set_ylabel("Screening score")
ax.set_title("Artificial-Lift Suitability for Mid-Water Tieback")
ax.grid(axis="y", alpha=0.3)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, score + 2, f"{score:.0f}", ha="center")
fig.savefig(FIGURES_DIR / "ch11_screening_artificial_lift.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch11_screening_artificial_lift.png).** *Observation.* Gas lift has the best screening score for the example because it combines reliability and operational flexibility. *Mechanism.* Gas lift tolerates moderate solids and water cut better than ESP while requiring less subsea rotating equipment than boosting. *Implication.* The lift method affects both production profile and intervention exposure. *Recommendation.* Screen lift method before freezing subsea controls and umbilical core counts.


## Feasibility Gate Register


In [4]:
gates = ["Host capacity", "Flowline hydraulics", "Hydrate control", "Materials", "Schedule"]
pass_fraction = np.array([0.80, 0.72, 0.65, 0.76, 0.70])
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(gates, pass_fraction, marker="o", linewidth=2, color="#2a9d8f")
ax.fill_between(gates, pass_fraction, 0.6, color="#2a9d8f", alpha=0.18)
ax.axhline(0.70, color="#e76f51", linestyle="--", label="DG1 minimum confidence")
ax.set_ylim(0.55, 0.9)
ax.set_ylabel("Gate confidence")
ax.set_title("Feasibility Gate Confidence Before Concept Select")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch11_screening_gate_confidence.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch11_screening_gate_confidence.png).** *Observation.* Hydrate control is below the DG1 confidence threshold and should remain an active risk. *Mechanism.* Flow-assurance mitigation needs fluid composition, water production, temperature and cooldown assumptions that are uncertain early. *Implication.* A concept can be economically attractive but still unready for concept select. *Recommendation.* Make the hydrate mitigation study a DG2 action with explicit owner, cost range and decision date.


## Exercises

1. Change the corrosion risk of the high-CO2 case from 5 to 3 and explain the design condition that justifies it.
2. Which gate has the weakest confidence, and what data would improve it?
3. Give two reasons why ESP can score lower than gas lift in subsea tiebacks.
4. Add an HSE gate and define a pass/fail criterion.
5. Explain how a failed feasibility gate should affect the MCDA score in the previous notebook.
